# 02 — Feature Engineering

Compute microstructure features: OFI (multi-level), VPIN, book imbalance,
weighted mid-price, spread, Kyle's lambda, trade flow aggression,
cancellation ratio, realized volatility, and return autocorrelation.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dashboard._mock_data import generate_snapshots
from src.features import (
    compute_ofi, compute_vpin, compute_book_imbalance,
    compute_weighted_mid, compute_spread_bps, compute_kyles_lambda,
    compute_trade_flow_aggression, compute_cancellation_ratio,
    compute_realized_volatility, compute_return_autocorrelation,
    build_feature_matrix,
)

pd.set_option("display.max_columns", 50)

## 2.1 Generate Snapshots and Compute Features

In [ ]:
df = generate_snapshots(n_timestamps=3600, start="2025-01-15 09:00:00", freq="1s")
print(f"Snapshots shape: {df.shape}")

# Build the full feature matrix
features = build_feature_matrix(df)
print(f"Feature matrix shape: {features.shape}")
print(f"Feature columns: {list(features.columns)}")
features.describe()

## 2.2 OFI (Order Flow Imbalance) — Multi-Level

In [ ]:
ofi = compute_ofi(df)
print(f"OFI columns: {list(ofi.columns)}")

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=["OFI (Depth 1)", "OFI (Depth 5)", "OFI (Depth 10)"],
                    vertical_spacing=0.06)

for i, d in enumerate([1, 5, 10], 1):
    fig.add_trace(go.Scatter(x=df["timestamp"], y=ofi[f"ofi_{d}_zscore"],
                             mode="lines", name=f"OFI-{d} z-score",
                             line=dict(width=0.8)), row=i, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=i, col=1)

fig.update_layout(height=600, template="plotly_dark",
                  title_text="Order Flow Imbalance at Multiple Depths (z-scored)")
fig.show()

In [ ]:
# OFI cross-depth correlation
ofi_cols = ["ofi_1_zscore", "ofi_5_zscore", "ofi_10_zscore"]
ofi_corr = ofi[ofi_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=ofi_corr.values, x=ofi_cols, y=ofi_cols,
    colorscale="RdBu_r", zmid=0, text=np.round(ofi_corr.values, 3),
    texttemplate="%{text}", zmin=-1, zmax=1))
fig.update_layout(title="OFI Cross-Depth Correlation",
                  template="plotly_dark", height=400, width=500)
fig.show()

## 2.3 VPIN (Volume-Synchronized Probability of Informed Trading)

In [ ]:
vpin = compute_vpin(df)
vpin_clean = vpin.dropna()

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=["VPIN Time Series", "VPIN Distribution"],
                    vertical_spacing=0.12)

fig.add_trace(go.Scatter(x=df["timestamp"][:len(vpin_clean)], y=vpin_clean.values,
                         mode="lines", name="VPIN",
                         line=dict(color="#e67e22", width=1)), row=1, col=1)
fig.add_hline(y=0.5, line_dash="dash", line_color="red",
              annotation_text="Toxicity threshold", row=1, col=1)

fig.add_trace(go.Histogram(x=vpin_clean.values, nbinsx=60,
                           marker_color="#e67e22", opacity=0.7,
                           name="VPIN dist"), row=2, col=1)

fig.update_layout(height=600, template="plotly_dark",
                  title_text="VPIN — Flow Toxicity Indicator")
fig.show()

print(f"VPIN statistics:")
print(f"  Mean:   {vpin_clean.mean():.4f}")
print(f"  Median: {vpin_clean.median():.4f}")
print(f"  Std:    {vpin_clean.std():.4f}")
print(f"  % > 0.5 (toxic): {(vpin_clean > 0.5).mean():.2%}")

## 2.4 Spread, Book Imbalance, and Kyle's Lambda

In [ ]:
spread = compute_spread_bps(df)
imbalance = compute_book_imbalance(df)
kyle = compute_kyles_lambda(df)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=["Spread (bps)", "Book Imbalance", "Kyle's Lambda"],
                    vertical_spacing=0.06)

fig.add_trace(go.Scatter(x=df["timestamp"], y=spread, mode="lines",
                         name="Spread", line=dict(color="#9b59b6", width=0.8)),
              row=1, col=1)

fig.add_trace(go.Scatter(x=df["timestamp"], y=imbalance, mode="lines",
                         name="Imbalance", line=dict(color="#f39c12", width=0.8)),
              row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

fig.add_trace(go.Scatter(x=df["timestamp"], y=kyle, mode="lines",
                         name="Kyle's Lambda", line=dict(color="#1abc9c", width=0.8)),
              row=3, col=1)

fig.update_layout(height=650, template="plotly_dark",
                  title_text="Spread, Book Imbalance, and Price Impact")
fig.show()

## 2.5 Realized Volatility (Multi-Horizon)

In [ ]:
rvol = compute_realized_volatility(df)

fig = go.Figure()
colors = ["#3498db", "#2ecc71", "#e67e22", "#e74c3c"]
for i, col in enumerate(rvol.columns):
    fig.add_trace(go.Scatter(x=df["timestamp"], y=rvol[col], mode="lines",
                             name=col, line=dict(color=colors[i], width=1)))

fig.update_layout(title="Realized Volatility at Multiple Horizons",
                  xaxis_title="Time", yaxis_title="Realized Vol",
                  template="plotly_dark", height=400)
fig.show()

## 2.6 Feature Correlation Matrix

In [ ]:
# Select a subset of key features for a readable correlation matrix
key_features = [c for c in features.columns if not c.startswith("ret_autocorr_")]
# Further trim to avoid clutter
key_features = [c for c in key_features if not (c.startswith("ofi_") and "velocity" in c)]

corr = features[key_features].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=key_features, y=key_features,
    colorscale="RdBu_r", zmid=0, zmin=-1, zmax=1))
fig.update_layout(title="Feature Correlation Matrix",
                  template="plotly_dark", height=700, width=800)
fig.show()

## 2.7 Feature Distributions

In [ ]:
# Distribution of key features (post z-score standardization)
plot_features = ["ofi_1_zscore", "vpin", "book_imbalance", "spread_bps",
                 "kyles_lambda", "rvol_1s", "rvol_60s", "cancellation_ratio"]
# Filter to columns that exist
plot_features = [f for f in plot_features if f in features.columns]

n_feats = len(plot_features)
n_cols = 2
n_rows = (n_feats + 1) // 2

fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=plot_features,
                    vertical_spacing=0.08)

for i, feat in enumerate(plot_features):
    r, c = divmod(i, n_cols)
    fig.add_trace(go.Histogram(x=features[feat].values, nbinsx=60,
                               marker_color="#3498db", opacity=0.7,
                               showlegend=False),
                  row=r + 1, col=c + 1)

fig.update_layout(height=200 * n_rows, template="plotly_dark",
                  title_text="Feature Distributions (z-scored)")
fig.show()

## 2.8 Return Autocorrelation Structure

In [ ]:
autocorr = compute_return_autocorrelation(df)

# Average autocorrelation at each lag
avg_autocorr = {f"lag_{k}": autocorr[f"ret_autocorr_{k}"].mean() for k in range(1, 11)}

fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(1, 11)),
                     y=[avg_autocorr[f"lag_{k}"] for k in range(1, 11)],
                     marker_color="#3498db", opacity=0.8))
fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(title="Average Return Autocorrelation by Lag",
                  xaxis_title="Lag (seconds)", yaxis_title="Autocorrelation",
                  template="plotly_dark", height=400)
fig.show()

print("Average autocorrelation by lag:")
for k in range(1, 11):
    print(f"  Lag {k:>2}: {avg_autocorr[f'lag_{k}']:+.6f}")

## Summary

Key observations from feature engineering:

1. **OFI cross-depth correlation:** OFI at different depths (1, 5, 10) is correlated but not identical — deeper levels add incremental information about order flow dynamics.
2. **VPIN distribution:** Mostly concentrated in the 0.1–0.5 range (balanced flow), with occasional spikes above 0.5 indicating potential informed trading.
3. **Realized volatility clustering:** All horizons show clear volatility clustering — periods of high vol persist, providing structure for the HMM to exploit.
4. **Feature standardization:** After rolling z-score normalization, features are approximately zero-centered and unit-scaled, suitable for Gaussian HMM emission modeling.
5. **Feature matrix:** 30 features assembled with no NaN/inf values, ready for HMM fitting.